In [14]:
import os
import mysql.connector
import pandas as pd

# List of CSV files and corresponding table names
csv_files = [
    ("customers.csv", "customers"),
    ("orders.csv", "orders"),
    ("sellers.csv", "sellers"),
    ("products.csv", "products"),
    ("order_items.csv", "order_items"),
    ("geolocation.csv", "geolocation"),
    ("payments.csv", "payments"),
]

# Connect to MySQL DB
conn = mysql.connector.connect(
    host="localhost",  # Replace with 'your_host' if not local
    user="root",
    password="root1234@#",
    database="ecommerce",  # Replace with your actual database name
)

cursor = conn.cursor()

# Folder path containing the CSV files
folder_path = r"C:\Users\Admin\Desktop\e-commerce" # Replace with your actual folder path


# Function to map pandas dtypes to SQL data types
def get_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "DATETIME"
    else:
        return "TEXT"


# Loop through each CSV file to process and upload
for csv_file, table_name in csv_files:
    file_path = os.path.join(folder_path, csv_file)

    # Read CSV file
    df = pd.read_csv(file_path)

    # Handle missing values by replacing NaN with None
    df = df.where(pd.notnull(df), None)

    print(f"Processing {csv_file}")
    print(f"NaN Values before replacement:\n{df.isnull().sum()}\n")

    # Clean column names (replace spaces, dashes, and dots with underscores)
    df.columns = [
        col.replace(" ", "_").replace("-", "_").replace(".", "_")
        for col in df.columns
    ]

    # Generate the CREATE TABLE statement with appropriate data types
    columns = ", ".join(
        [f"`{col}` {get_sql_type(df[col].dtype)}" for col in df.columns]
    )
    create_table_query = (
        f"CREATE TABLE IF NOT EXISTS `{table_name}` ({columns})"
    )

    cursor.execute(create_table_query)

    # Insert DataFrame data into the MySQL table
    for _, row in df.iterrows():
        # Convert row to tuple and handle NaN/None explicitly
        values = tuple(None if pd.isna(x) else x for x in row)

        # Construct placeholders for SQL statement
        cols_str = ", ".join([f"`{col}`" for col in df.columns])
        placeholders = ", ".join(["%s"] * len(df.columns))

        sql = f"INSERT INTO `{table_name}` ({cols_str}) VALUES ({placeholders})"
        cursor.execute(sql, values)

    # Commit the transaction for the current CSV file
    conn.commit()

# Close the connection
cursor.close()
conn.close()
print("Data migration completed successfully!")

Processing customers.csv
NaN Values before replacement:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Processing orders.csv
NaN Values before replacement:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Processing sellers.csv
NaN Values before replacement:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Processing products.csv
NaN Values before replacement:
product_id                      0
product category              610
product_name_length           610
product_description_length    610
product_photos_qty            610
prod

#### pip install mysql-connector-python;
